# 0 - COLAB SETUP
Chạy **cell ngay dưới đây ĐẦU TIÊN** khi train trên Google Colab.
Nó sẽ: mount Google Drive (lưu bền vững), tạo cây thư mục, tải dataset từ Kaggle,
và trỏ `checkpoints/`, `outputs/` sang Drive để **không mất khi session reset**.
Chạy ở máy local thì cell tự bỏ qua.

In [ ]:
# ============================================================
#  COLAB SETUP  (chay DAU TIEN tren Google Colab)
# ============================================================
import os, sys, glob, shutil, subprocess, collections

IS_COLAB = "google.colab" in sys.modules

# ===== CHI CAN SUA 2 DONG DUOI =====
KAGGLE_DATASET = "kienngx/callhome-dataset"     # slug "owner/ten" HOAC dan ca URL cung duoc
DRIVE_DIR      = "/content/drive/MyDrive/eend"   # noi luu checkpoint + output (ben vung)
# ===================================

# Tu dong cat slug neu lo dan ca URL (https://www.kaggle.com/datasets/owner/ten)
KAGGLE_DATASET = KAGGLE_DATASET.strip().rstrip("/")
if "kaggle.com" in KAGGLE_DATASET:
    after = KAGGLE_DATASET.split("kaggle.com/datasets/")[-1].split("kaggle.com/")[-1]
    KAGGLE_DATASET = "/".join(after.split("/")[:2])
print("Dataset slug:", KAGGLE_DATASET)

# Giu nguyen path tuyet doi de khop TOAN BO notebook (khong can sua cell khac)
PROJECT_DIR = "/home/ju1ian/Documents/eend"

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    for sub in ["checkpoints", "outputs"]:
        os.makedirs(f"{DRIVE_DIR}/{sub}", exist_ok=True)

    for sub in ["models", "losses", "data/processed", "CallHome/audio", "CallHome/labels"]:
        os.makedirs(f"{PROJECT_DIR}/{sub}", exist_ok=True)

    for name in ["checkpoints", "outputs"]:
        link = f"{PROJECT_DIR}/{name}"
        if os.path.islink(link):
            os.unlink(link)
        elif os.path.isdir(link):
            shutil.rmtree(link)
        os.symlink(f"{DRIVE_DIR}/{name}", link)

    subprocess.run("pip install -q librosa scikit-learn kaggle".split())

    os.makedirs("/root/.kaggle", exist_ok=True)
    kj = "/root/.kaggle/kaggle.json"
    if not os.path.exists(kj):
        if os.path.exists(f"{DRIVE_DIR}/kaggle.json"):
            shutil.copy(f"{DRIVE_DIR}/kaggle.json", kj)
        else:
            print(">> Upload file kaggle.json (Kaggle > Settings > Create Legacy API Key):")
            from google.colab import files
            files.upload()
            shutil.copy("kaggle.json", kj)
            shutil.copy("kaggle.json", f"{DRIVE_DIR}/kaggle.json")
    os.chmod(kj, 0o600)

    if not glob.glob(f"{PROJECT_DIR}/CallHome/audio/*.wav"):
        DL = "/content/kaggle_ds"
        r = subprocess.run(["kaggle", "datasets", "download", "-d", KAGGLE_DATASET,
                            "-p", DL, "--unzip"], capture_output=True, text=True)
        print(r.stdout); print(r.stderr)
        if r.returncode != 0:
            raise RuntimeError(f"Tai dataset that bai. Kiem tra slug '{KAGGLE_DATASET}' "
                               f"(xem: kaggle datasets files -d {KAGGLE_DATASET}).")
        all_files = [f for f in glob.glob(f"{DL}/**/*", recursive=True) if os.path.isfile(f)]
        wavs  = [f for f in all_files if f.lower().endswith(".wav")]
        rttms = [f for f in all_files if f.lower().endswith(".rttm")]
        print(f"  Tim thay {len(wavs)} .wav, {len(rttms)} .rttm")
        if not wavs or not rttms:
            ext = collections.Counter(os.path.splitext(f)[1].lower() for f in all_files)
            print("\n!! Khong thay .wav hoac .rttm. Dataset chua cac duoi file:")
            print("  ", ext.most_common(15))
            for f in all_files[:15]:
                print("   ", f.replace(DL, ""))
            raise RuntimeError("Cau truc/duoi file khac mong doi.")
        for w in wavs:
            dst = f"{PROJECT_DIR}/CallHome/audio/{os.path.basename(w)}"
            if not os.path.exists(dst): os.symlink(w, dst)
        for r2 in rttms:
            dst = f"{PROJECT_DIR}/CallHome/labels/{os.path.basename(r2)}"
            if not os.path.exists(dst): os.symlink(r2, dst)

    na = len(glob.glob(f"{PROJECT_DIR}/CallHome/audio/*.wav"))
    nl = len(glob.glob(f"{PROJECT_DIR}/CallHome/labels/*.rttm"))
    print(f"\nColab setup xong | audio={na} labels={nl} | checkpoints+outputs -> {DRIVE_DIR}")
    assert na > 0 and nl > 0, "Chua thay du lieu! Kiem tra lai KAGGLE_DATASET hoac kaggle.json."
else:
    print("Chay o may local — bo qua Colab setup.")


# I - Folder Structure & Requirements.txt


In [1]:
%pip install jupyter-cache  

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
base = "/home/ju1ian/Documents/eend"
folders = [
    "data/raw",
    "data/processed",
    "models",
    "losses",
    "checkpoints",
    "outputs",
    
]

for f in folders:
  os.makedirs(os.path.join(base, f), exist_ok=True)

print("Folder structure created: ")
for f in folders:
  print(f" EEDN/{f}")

Folder structure created: 
 EEDN/data/raw
 EEDN/data/processed
 EEDN/models
 EEDN/losses
 EEDN/checkpoints
 EEDN/outputs


*GPU CHECK*

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


if device.type == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
else:
    print("  No GPU found — go to Runtime > Change runtime type > GPU")

Using device: cuda
  GPU: NVIDIA GeForce GTX 1650
  VRAM: 3.9 GB


# II - DATASET LOADING


> **AUDIO LOADING**

In [4]:
import librosa
import os 
import numpy as np

# SR = 16000

wav_folder = "/home/ju1ian/Documents/eend/CallHome/audio"

wav_files = sorted([
    os.path.join(wav_folder, f)
    for f in os.listdir(wav_folder)
    if f.endswith(".wav")
])

print(f"found {len(wav_files)}.WAV files")

audio_arrays = [
    librosa.load(path, sr=16000)[0]
    for path in wav_files
]

print(f"loaded {len(audio_arrays)} arrays")


found 140.WAV files


/home/ju1ian/miniconda3/envs/eend-vd/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded 140 arrays


> **LABEL LOADING**

In [5]:
import os 

def parse_rttm_file(rttm_file_path):
    # Initialize the empty lists your function needs
    timestamps_start = []
    timestamps_end = []
    speakers = []
    
    # Open and read the text file
    with open(rttm_file_path, 'r') as file:
        for line in file:
            # Strip extra whitespace and split the line by spaces
            parts = line.strip().split()
            
            # Make sure it's a valid SPEAKER line (it should have at least 8 columns)
            if len(parts) >= 8 and parts[0] == "SPEAKER":
                
                # Extract the relevant columns (converting numbers to floats)
                start_time = float(parts[3])
                duration = float(parts[4])
                speaker_id = parts[7]
                
                # Calculate the end time
                end_time = start_time + duration
                
                # Add the extracted data to our lists
                timestamps_start.append(start_time)
                timestamps_end.append(end_time)
                speakers.append(speaker_id)
                
    return timestamps_start, timestamps_end, speakers

labels_path = "/home/ju1ian/Documents/eend/CallHome/labels"

rttm_files = sorted([
    os.path.join(labels_path, f)
    for f in os.listdir(labels_path)
    if f.endswith(".rttm")
     
])
# print(rttm_files)

results = [
    parse_rttm_file(path)
    for path in rttm_files
]

starts, ends, spks = zip(*results)


--------------------------------------------------------------


> **SAMPLE VISUALIZING**

In [ ]:
import matplotlib.pyplot as plt

speakers_list = spks[0]
starts_list = starts[0]
ends_list = ends[0]


speakers = list(set(speakers_list))
speaker_to_int = {s: i for i, s in enumerate(speakers)}
colors = ["steelblue", "tomato"]

fig, ax = plt.subplots(figsize=(14 ,2))
for start, end, spk in zip(starts_list,
                           ends_list,
                           speakers_list):
  ax.barh(
      speaker_to_int[spk], end - start,
      left = start, height = 0.4,
      color = colors[speaker_to_int[spk] % len(colors)]
  )

ax.set_yticks(range(len(speakers)))
ax.set_yticklabels(speakers)
ax.set_xlabel("Time (seconds)")
ax.set_title("Speaker Timeline - Sample 0")
plt.tight_layout()
import os; os.makedirs("/home/ju1ian/Documents/eend/outputs", exist_ok=True)
plt.savefig("/home/ju1ian/Documents/eend/outputs/sample_timeline.png", dpi=150, bbox_inches="tight")  # luu vao outputs/ (Drive tren Colab)
plt.show()

# III - DATA PREPROCESSING



> **PARAMETERS CONFIG**



In [7]:
#Configs
SAMPLE_RATE = 16000       # CALLHOME Fre is 16kHz
N_MELS = 40               # Number of mel filterbanks
FRAME_SIZE = 25           # millisec
FRAME_SHIFT = 10          # millisec
SUBSAMPLING = 10          # stack every N frames together


FRAME_SIZE_SAMPLES = int(SAMPLE_RATE * FRAME_SIZE / 1000)
FRAME_SIZE_SHIFT = int(SAMPLE_RATE * FRAME_SHIFT / 1000)

print("Preprocessing configs: ")
print(f" Sameple Rate     : {SAMPLE_RATE}")
print(f" Mel Filterbanks  : {N_MELS}")
print(f" Frame Size       : {FRAME_SIZE}ms ({FRAME_SIZE_SAMPLES})ms")
print(f" FRAME_SHIFT      : {FRAME_SHIFT}ms ({FRAME_SIZE_SHIFT})ms")
print(f" Subsampling      : {SUBSAMPLING} frames")

Preprocessing configs: 
 Sameple Rate     : 16000
 Mel Filterbanks  : 40
 Frame Size       : 25ms (400)ms
 FRAME_SHIFT      : 10ms (160)ms
 Subsampling      : 10 frames


--------------------------------------------------------------




> **AUDIO -> LOG-MEL SPECTROGRAM**



In [8]:
import numpy as np
import librosa

def extract_logmel(audio_array, sample_rate=SAMPLE_RATE):
  """


  """

  mel = librosa.feature.melspectrogram(
      y = audio_array,
      sr = sample_rate,
      n_mels = N_MELS,
      hop_length = FRAME_SIZE_SHIFT,
      n_fft = FRAME_SIZE_SAMPLES,
  )

  logmel = librosa.power_to_db(mel, ref=np.max)
  return logmel.T

--------------------------------------------------------------





> **TIMESTAMP -> FRAME-LEVEL SPEAKER LABEl**





In [9]:
FRAME_SHIFT_SEC = 0.01

def make_speaker_labels(timestamps_start, timestamps_end, speakers, num_frames, max_speakers=4):
  """



  """

  unique_speakers = sorted(set(speakers))
  speaker_to_idx = {s: i for i, s in enumerate(unique_speakers)}
  num_speakers = len(unique_speakers)
  labels = np.zeros((num_frames, num_speakers), dtype=np.float32)

  for start, end, spk in zip(timestamps_start, timestamps_end, speakers):
      start_frame = int(start / FRAME_SHIFT_SEC)
      end_frame = int(end / FRAME_SHIFT_SEC)
      start_frame = max(0, min(start_frame, num_frames - 1))
      end_frame = max(0, min(end_frame, num_frames))
      spk_idx = speaker_to_idx[spk]
      if spk_idx < max_speakers:
        labels[start_frame:end_frame, spk_idx] = 1.0


  return labels, unique_speakers

# ALL in one block — no stale variables
print(audio_arrays[0])
features = extract_logmel(audio_arrays[0])
num_frames = features.shape[0]  # freshly computed here
print(num_frames)

labels, spk_list = make_speaker_labels(
  starts[0],
  ends[0],
  spks[0],
  num_frames
)

print(f"num_frames      : {num_frames}")
print(f"labels shape    : {labels.shape}")
print(f"active frames   : {labels.sum(axis=0)}")
print(f"speakers        : {spk_list}")
print()


print("Label Generation Works!")
print(f" Speakers found   : {spk_list}")
print(f" Labels shape     : {labels.shape} (frames x speakers)")
print(f" Features shape   : {features.shape} (frames x mels)")
print(f" Frames match     : {features.shape[0] == labels.shape[0]}")

[-0.00689697  0.02844238  0.04931641 ... -0.00048828 -0.00085449
 -0.00073242]
30313
num_frames      : 30313
labels shape    : (30313, 2)
active frames   : [18814.  7177.]
speakers        : ['A', 'B']

Label Generation Works!
 Speakers found   : ['A', 'B']
 Labels shape     : (30313, 2) (frames x speakers)
 Features shape   : (30313, 40) (frames x mels)
 Frames match     : True


--------------------------------------------------------------


In [10]:
import shutil, os, numpy as np, librosa
from tqdm import tqdm

CACHE_DIR     = "/home/ju1ian/Documents/eend/data/processed"
FRAME_SHIFT_SEC = 0.01

# Clear old cache completely
shutil.rmtree(CACHE_DIR)
os.makedirs(CACHE_DIR)
print("🗑️  Old cache cleared")

def _extract_logmel(audio_array):
    mel = librosa.feature.melspectrogram(
        y=audio_array.astype(np.float32), sr=16000,
        n_mels=40, n_fft=400, hop_length=160
    )
    return librosa.power_to_db(mel, ref=np.max).T

def _make_labels(ts_start, ts_end, speakers, num_frames, max_speakers=4):
    spk_map = {s: i for i, s in enumerate(sorted(set(speakers)))}
    labels  = np.zeros((num_frames, max_speakers), dtype=np.float32)
    for s, e, spk in zip(ts_start, ts_end, speakers):
        sf = max(0, min(int(s / FRAME_SHIFT_SEC), num_frames - 1))
        ef = max(0, min(int(e / FRAME_SHIFT_SEC), num_frames))
        if spk_map[spk] < max_speakers:
            labels[sf:ef, spk_map[spk]] = 1.0
    return labels

print("Caching all samples with max_speakers=4...")
for idx in tqdm(range(len(audio_arrays))):
    sample     = audio_arrays[0]
    features   = _extract_logmel(audio_arrays[idx])
    num_frames = features.shape[0]
    labels     = _make_labels(
        starts[idx], ends[idx],
        spks[idx], num_frames
    )
    assert labels.shape == (num_frames, 4), f"Wrong shape at idx={idx}: {labels.shape}"
    np.save(os.path.join(CACHE_DIR, f"feat_{idx}.npy"),  features)
    np.save(os.path.join(CACHE_DIR, f"label_{idx}.npy"), labels)

print("✅ All cached and verified!")

print(labels[1])

🗑️  Old cache cleared
Caching all samples with max_speakers=4...


100%|██████████| 140/140 [00:22<00:00,  6.12it/s]

✅ All cached and verified!
[0. 1. 0. 0.]




> Visualize features + labels together





In [ ]:

import matplotlib.pyplot as plt

SHOW_FRAMES = 2000

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

# Top: log-mel spectrogram
axes[0].imshow(features[:SHOW_FRAMES].T, aspect="auto", origin="lower", cmap="viridis")
axes[0].set_ylabel("Mel bins")
axes[0].set_title(f"Log-Mel Spectrogram (first {SHOW_FRAMES} frames)")

# Bottom: speaker activity
colors = ["steelblue", "tomato", "seagreen", "orange"]
for i, spk in enumerate(spk_list):
    axes[1].fill_between(
        range(SHOW_FRAMES),
        i, i + 0.8,
        where=labels[:SHOW_FRAMES, i] > 0,
        color=colors[i % len(colors)],
        label=spk
    )
axes[1].set_yticks(range(len(spk_list)))
axes[1].set_yticklabels(spk_list)
axes[1].set_xlabel("Frame index")
axes[1].set_title("Speaker Activity Labels")
axes[1].legend(loc="upper right")

plt.tight_layout()
import os; os.makedirs("/home/ju1ian/Documents/eend/outputs", exist_ok=True)
plt.savefig("/home/ju1ian/Documents/eend/outputs/logmel_labels.png", dpi=150, bbox_inches="tight")  # luu vao outputs/ (Drive tren Colab)
plt.show()

# IV - Split & Dataset Class

In [12]:
from sklearn.model_selection import train_test_split

all_indices = list(range(len(audio_arrays)))
print(f"Total Samples: {len(all_indices)}")

train_val_indices, test_indices = train_test_split(
  all_indices, test_size=0.1, random_state=42
)

train_indices, val_indices = train_test_split(
    train_val_indices, test_size=0.111, random_state=42
)

print(f"Train samples: {train_val_indices}")
print(f"Val samples  : {val_indices}")
print(f"Test samples : {test_indices}")


Total Samples: 140
Train samples: [132, 78, 11, 27, 127, 110, 36, 118, 60, 4, 131, 26, 138, 96, 16, 18, 10, 111, 101, 94, 51, 45, 82, 136, 65, 0, 55, 28, 40, 24, 93, 126, 112, 64, 44, 15, 89, 39, 22, 105, 76, 100, 47, 30, 83, 73, 9, 33, 77, 25, 86, 62, 68, 135, 53, 5, 80, 85, 49, 35, 34, 97, 7, 43, 70, 84, 117, 123, 8, 13, 133, 3, 17, 38, 72, 114, 6, 95, 125, 129, 54, 50, 98, 46, 128, 61, 137, 79, 115, 91, 41, 58, 90, 48, 88, 120, 124, 57, 75, 32, 134, 59, 63, 122, 37, 29, 107, 130, 1, 52, 21, 2, 23, 103, 99, 116, 87, 74, 121, 139, 20, 71, 106, 14, 92, 102]
Val samples  : [38, 94, 87, 123, 88, 95, 126, 135, 74, 83, 127, 37, 89, 57]
Test samples : [108, 67, 31, 119, 42, 12, 81, 69, 104, 109, 113, 56, 66, 19]



--------------------------------------------------------------




> PyTorch Dataset Class



In [13]:
%%writefile /home/ju1ian/Documents/eend/data/dataset.py

import torch
from torch.utils.data import Dataset
import numpy as np
import os

CACHE_DIR = "/home/ju1ian/Documents/eend/data/processed"

class CallHomeDataset(Dataset):
    """
    augment=True  -> bật SpecAugment (chỉ dùng cho tập train) để chống overfit.
    freq_mask/time_mask: độ rộng tối đa của dải mask; num_masks: số dải mỗi loại.
    """
    def __init__(self, indices, chunk_size=500, augment=False,
                 freq_mask=8, time_mask=50, num_masks=2):
        self.indices    = indices
        self.chunk_size = chunk_size
        self.augment    = augment
        self.freq_mask  = freq_mask
        self.time_mask  = time_mask
        self.num_masks  = num_masks
        self.chunks     = self._build_chunks()

    def _build_chunks(self):
        chunks = []
        for idx in self.indices:
            feat       = np.load(os.path.join(CACHE_DIR, f"feat_{idx}.npy"))
            num_frames = feat.shape[0]
            for start in range(0, num_frames - self.chunk_size, self.chunk_size):
                chunks.append((idx, start))
        return chunks

    def _spec_augment(self, feat):
        # feat: (chunk_size, n_mels)
        feat = feat.copy()
        T, F = feat.shape
        for _ in range(self.num_masks):
            # time mask
            t = np.random.randint(0, self.time_mask + 1)
            if t > 0 and T - t > 0:
                t0 = np.random.randint(0, T - t)
                feat[t0:t0 + t, :] = 0.0
            # frequency mask
            f = np.random.randint(0, self.freq_mask + 1)
            if f > 0 and F - f > 0:
                f0 = np.random.randint(0, F - f)
                feat[:, f0:f0 + f] = 0.0
        return feat

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, i):
        idx, start = self.chunks[i]
        feat  = np.load(os.path.join(CACHE_DIR, f"feat_{idx}.npy"))[start:start + self.chunk_size]
        label = np.load(os.path.join(CACHE_DIR, f"label_{idx}.npy"))[start:start + self.chunk_size]
        if self.augment:
            feat = self._spec_augment(feat)
        return (
            torch.tensor(feat,  dtype=torch.float32),
            torch.tensor(label, dtype=torch.float32),
        )

print("dataset.py written (with SpecAugment)")


Overwriting /home/ju1ian/Documents/eend/data/dataset.py


# V - MODEL

Enconder (BLSTM)

In [14]:
%%writefile /home/ju1ian/Documents/eend/models/encoder.py

# import torch
# import torch.nn as nn

# class BLSTMEncoder(nn.Module):
#     def __init__(self, input_size=40, hidden_size=256, num_layers=4, dropout=0.3):
#         """
#         input_size  : number of mel bins (40)
#         hidden_size : LSTM hidden units per direction
#         num_layers  : number of stacked BLSTM layers
#         dropout     : dropout between layers
#         """
#         super().__init__()
#         self.lstm = nn.LSTM(
#             input_size=input_size,
#             hidden_size=hidden_size,
#             num_layers=num_layers,
#             batch_first=True,       # input shape: (batch, frames, features)
#             bidirectional=True,
#             dropout=dropout if num_layers > 1 else 0.0
#         )
#         self.output_size = hidden_size * 2  # ×2 because bidirectional

#     def forward(self, x):
#         """
#         x   : (batch, frames, input_size)
#         out : (batch, frames, hidden_size * 2)
#         """
#         out, _ = self.lstm(x)
#         return out


# %%writefile /home/ju1ian/Documents/eend/models/encoder.py

import torch
import torch.nn as nn

class BLSTMEncoder(nn.Module):
    def __init__(self, input_size=40, hidden_size=256, num_layers=4, dropout=0.1):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        # Thêm LayerNorm để ổn định huấn luyện
        self.ln = nn.LayerNorm(hidden_size * 2)
        self.output_size = hidden_size * 2

    def forward(self, x):
        # x : (batch, frames, input_size)
        out, _ = self.lstm(x)
        out = self.ln(out) 
        return out

Overwriting /home/ju1ian/Documents/eend/models/encoder.py




> Attractor



In [15]:
%%writefile /home/ju1ian/Documents/eend/models/attractor.py

import torch
import torch.nn as nn

class AttractorModule(nn.Module):
    """
    LSTM decoder sinh tuan tu cac attractor.
    De DEM so nguoi noi (EDA): luc train se sinh num_speakers + 1 attractor,
    attractor cuoi cung dong vai tro "STOP" (existence target = 0).
    """
    def __init__(self, encoder_size=512, attractor_size=256):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=encoder_size,
            hidden_size=attractor_size,
            batch_first=True,
        )
        self.linear = nn.Linear(attractor_size, 1)   # existence LOGIT (chua sigmoid)
        # Projection de khoi tao hidden state tu encoder (512 -> 256)
        self.initial_proj = nn.Linear(encoder_size, attractor_size)

    def forward(self, encoder_out, num_attractors):
        """
        encoder_out    : (batch, frames, encoder_size)
        num_attractors : so attractor can sinh

        Returns:
            attractors       : (batch, num_attractors, attractor_size)
            existence_logits : (batch, num_attractors, 1)
        """
        batch_size = encoder_out.size(0)
        encoder_size = encoder_out.size(-1)

        # Khoi tao h0, c0 tu trung binh encoder out
        mean_enc = encoder_out.mean(dim=1)                              # (batch, encoder_size)
        h = self.initial_proj(mean_enc).unsqueeze(0).contiguous()      # (1, batch, attractor_size)
        c = torch.zeros_like(h)

        dummy_input = torch.zeros(batch_size, 1, encoder_size, device=encoder_out.device)

        attractors, existence = [], []
        for _ in range(num_attractors):
            out, (h, c) = self.lstm(dummy_input, (h, c))
            attractors.append(out)                  # (batch, 1, attractor_size)
            existence.append(self.linear(out))      # (batch, 1, 1)

        attractors = torch.cat(attractors, dim=1)   # (batch, num_attractors, attractor_size)
        existence  = torch.cat(existence,  dim=1)   # (batch, num_attractors, 1)
        return attractors, existence


Overwriting /home/ju1ian/Documents/eend/models/attractor.py




> EEND main model



In [16]:
%%writefile /home/ju1ian/Documents/eend/models/eend.py

import torch
import torch.nn as nn
import sys
sys.path.append("/home/ju1ian/Documents/eend/models")

from encoder import BLSTMEncoder
from attractor import AttractorModule

class EEND(nn.Module):
    def __init__(self, input_size=40, hidden_size=256, num_layers=4,
                 dropout=0.3, attractor_size=256):
        super().__init__()

        self.encoder = BLSTMEncoder(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout,
        )
        self.attractor = AttractorModule(
            encoder_size=self.encoder.output_size,   # 2*hidden = 512
            attractor_size=attractor_size,
        )
        self.projection = nn.Linear(self.encoder.output_size, attractor_size)

    def forward(self, x, num_speakers=4):
        """
        Sinh num_speakers (= S_max) attractor.
        existence_logits cua tung attractor duoc dung de DEM speaker:
        attractor ung voi speaker that -> p->1, attractor du thua -> p->0.

        Returns:
            logits           : (batch, frames, num_speakers)
            existence_logits : (batch, num_speakers, 1)
        """
        enc_out  = self.encoder(x)            # (batch, frames, 512)
        enc_proj = self.projection(enc_out)   # (batch, frames, attractor_size)

        attractors, existence_logits = self.attractor(enc_out, num_speakers)
        logits = torch.bmm(enc_proj, attractors.transpose(1, 2))  # (batch, frames, num_speakers)
        return logits, existence_logits


Overwriting /home/ju1ian/Documents/eend/models/eend.py


In [17]:
import sys
sys.path.append("/home/ju1ian/Documents/eend/models")
from eend import EEND
import torch

# Init model
model = EEND(
    input_size=40,
    hidden_size=256,
    num_layers=4,
    dropout=0.3,
    attractor_size=256,
)

# Dummy forward pass
batch_size  = 4
num_frames  = 500
num_speakers = 4

dummy_input = torch.randn(batch_size, num_frames, 40)
logits, existence_probs = model(dummy_input, num_speakers=num_speakers)

print(f"✅ Model forward pass works!")
print(f"  Input shape          : {dummy_input.shape}")
print(f"  Logits shape         : {logits.shape}")         # (4, 500, 2)
print(f"  Existence probs shape: {existence_probs.shape}") # (4, 2, 1)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

✅ Model forward pass works!
  Input shape          : torch.Size([4, 500, 40])
  Logits shape         : torch.Size([4, 500, 4])
  Existence probs shape: torch.Size([4, 4, 1])

Total parameters: 6,393,601


# VI - Losses

In [18]:
%%writefile /home/ju1ian/Documents/eend/losses/pit_loss.py

import torch
import torch.nn as nn
from itertools import permutations

class PITLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # reduction="none" -> chon permutation tot nhat cho TUNG sample.
        self.bce = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, logits, labels):
        """
        logits : (batch, frames, num_speakers)
        labels : (batch, frames, num_speakers)

        Returns:
            loss      : scalar (trung binh tren batch)
            best_perm : (batch, num_speakers) - best_perm[b][j] = chi so attractor
                        chiu trach nhiem cho cot label j cua sample b.
                        (Dung de gan existence target trong AttractorLoss.)
        """
        batch_size, frames, num_speakers = logits.shape
        perms = list(permutations(range(num_speakers)))

        losses = []
        for perm in perms:
            perm_logits = logits[:, :, perm]                       # (B, T, S)
            loss = self.bce(perm_logits, labels).mean(dim=(1, 2))  # (B,)
            losses.append(loss)

        losses = torch.stack(losses, dim=1)                        # (B, num_perms)
        min_loss, min_idx = losses.min(dim=1)                      # (B,)

        perm_tensor = torch.tensor(perms, device=logits.device, dtype=torch.long)  # (num_perms, S)
        best_perm = perm_tensor[min_idx]                           # (B, S)

        return min_loss.mean(), best_perm


Overwriting /home/ju1ian/Documents/eend/losses/pit_loss.py




> Attractor Loss



In [19]:
%%writefile /home/ju1ian/Documents/eend/losses/attractor_loss.py

import torch
import torch.nn as nn

class AttractorLoss(nn.Module):
    """
    Day model DEM so nguoi noi (eq. Attractor Loss trong tieu luan).
    Nhan ton tai q_s cho attractor thu s:
        q_s = 1 neu attractor khop voi 1 speaker THAT su co hoat dong
        q_s = 0 voi attractor du thua (cot label rong)
    Viec attractor nao khop speaker nao lay tu hoan vi toi uu phi* cua PIT.
    """
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(reduction="mean")

    def forward(self, existence_logits, labels, best_perm, num_speakers):
        """
        existence_logits : (batch, num_speakers, 1)
        labels           : (batch, frames, num_speakers)
        best_perm        : (batch, num_speakers) tu PITLoss
        """
        batch_size = existence_logits.size(0)
        device = existence_logits.device

        # Cot label nao thuc su co hoat dong trong chunk nay (= so speaker S)
        active = (labels.sum(dim=1) > 0).float()                 # (B, S_max)

        # Gan theo hoan vi: q[b, best_perm[b][j]] = active[b][j]
        q = torch.zeros(batch_size, num_speakers, device=device)
        q.scatter_(1, best_perm, active)                         # (B, S_max)

        return self.bce(existence_logits.squeeze(-1), q)


Overwriting /home/ju1ian/Documents/eend/losses/attractor_loss.py




> Combined Loss



In [20]:
%%writefile /home/ju1ian/Documents/eend/losses/combined_loss.py

import sys
sys.path.append("/home/ju1ian/Documents/eend/losses")
import torch.nn as nn
from pit_loss import PITLoss
from attractor_loss import AttractorLoss

class EENDLoss(nn.Module):
    def __init__(self, attractor_weight=1.0):
        super().__init__()
        self.pit_loss        = PITLoss()
        self.attractor_loss  = AttractorLoss()
        self.attractor_weight = attractor_weight

    def forward(self, logits, labels, existence_logits, num_speakers):
        """
        logits           : (batch, frames, num_speakers)
        labels           : (batch, frames, num_speakers)
        existence_logits : (batch, num_speakers + 1, 1)
        """
        pit, best_perm = self.pit_loss(logits, labels)
        attr = self.attractor_loss(existence_logits, labels, best_perm, num_speakers)
        total = pit + self.attractor_weight * attr
        return total, pit, attr


Overwriting /home/ju1ian/Documents/eend/losses/combined_loss.py




> Test all losses



In [21]:
import sys
sys.path.append("/home/ju1ian/Documents/eend/losses")
sys.path.append("/home/ju1ian/Documents/eend/models")

# Force clean imports
for mod in ["pit_loss", "attractor_loss", "combined_loss"]:
    if mod in sys.modules:
        del sys.modules[mod]

from combined_loss import EENDLoss
from eend import EEND
import torch

# Setup
model    = EEND(input_size=40, hidden_size=256, num_layers=4,
                dropout=0.3, attractor_size=256)
criterion = EENDLoss(attractor_weight=1.0)

# Dummy data
dummy_input  = torch.randn(4, 500, 40)
dummy_labels = torch.randint(0, 2, (4, 500, 2)).float()

# Forward pass
logits, existence_probs = model(dummy_input, num_speakers=2)
total_loss, pit_loss, attr_loss = criterion(
    logits, dummy_labels, existence_probs, num_speakers=2
)

print("✅ Loss computation works!")
print(f"  Total loss     : {total_loss.item():.4f}")
print(f"  PIT loss       : {pit_loss.item():.4f}")
print(f"  Attractor loss : {attr_loss.item():.4f}")

✅ Loss computation works!
  Total loss     : 1.3540
  PIT loss       : 0.7113
  Attractor loss : 0.6428


# VI - Trainning Loop

In [22]:
CONFIG = {
   "input_size"         : 40,
   "hidden_size"        : 256,    # giam tu 512 -> chong overfit
   "num_layers"         : 4,      # giam tu 6   -> chong overfit
   "dropout"            : 0.3,    # tang tu 0.1
   "attractor_size"     : 256,
   "num_speakers"       : 4,
   "chunk_size"         : 500,
   "num_epochs"         : 50,     # nhieu hon nhung co early stopping
   "batch_size"         : 16,   # GTX1650 4GB du suc, batch=8 chi ~0.46GB
   "learning_rate"      : 1e-4,
   "weight_decay"       : 1e-2,   # AdamW regularization
   "attractor_weight"   : 1.0,
   "checkpoint_dir"     : "/home/ju1ian/Documents/eend/checkpoints",
   "patience"           : 8
}

print("config set:")
for k, v in CONFIG.items():
  print(f"  {k:20s}: {v}")


config set:
  input_size          : 40
  hidden_size         : 256
  num_layers          : 4
  dropout             : 0.3
  attractor_size      : 256
  num_speakers        : 4
  chunk_size          : 500
  num_epochs          : 50
  batch_size          : 16
  learning_rate       : 0.0001
  weight_decay        : 0.01
  attractor_weight    : 1.0
  checkpoint_dir      : /home/ju1ian/Documents/eend/checkpoints
  patience            : 8


> Setup, optimizer, dataloaders


In [23]:
import sys
import torch
from torch.utils.data import DataLoader

sys.path.append("/home/ju1ian/Documents/eend/models")
sys.path.append("/home/ju1ian/Documents/eend/losses")
sys.path.append("/home/ju1ian/Documents/eend/data")

# Clear all cached modules
for mod in ["eend", "encoder", "attractor", "combined_loss",
            "pit_loss", "attractor_loss", "dataset"]:
    if mod in sys.modules:
        del sys.modules[mod]

# Now import cleanly
from eend import EEND
from combined_loss import EENDLoss
from dataset import CallHomeDataset

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Model
model = EEND(
    input_size     = CONFIG["input_size"],
    hidden_size    = CONFIG["hidden_size"],
    num_layers     = CONFIG["num_layers"],
    dropout        = CONFIG["dropout"],
    attractor_size = CONFIG["attractor_size"],
).to(device)

# Loss & optimizer (AdamW + weight decay)
criterion = EENDLoss(attractor_weight=CONFIG["attractor_weight"])
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

# Giam LR khi val loss khong cai thien
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)

# Dataloaders (chi augment tap train)
train_loader = DataLoader(
    CallHomeDataset(train_indices, chunk_size=CONFIG["chunk_size"], augment=True),
    batch_size=CONFIG["batch_size"], shuffle=True, num_workers=2
)
val_loader = DataLoader(
    CallHomeDataset(val_indices, chunk_size=CONFIG["chunk_size"], augment=False),
    batch_size=CONFIG["batch_size"], shuffle=False, num_workers=2
)

print("Setup complete")
print(f"  Train batches : {len(train_loader)}")
print(f"  Val batches   : {len(val_loader)}")
print(f"  Model params  : {sum(p.numel() for p in model.parameters()):,}")


dataset.py written (with SpecAugment)
Using device: cuda
Setup complete
  Train batches : 739
  Val batches   : 72
  Model params  : 6,393,601




> Training Loop



In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = pit_total = attr_total = 0

    for batch_idx, (feats, labels) in enumerate(loader):
        feats  = feats.to(device)
        labels = labels.to(device)

        logits, existence_probs = model(feats, num_speakers=CONFIG["num_speakers"])
        loss, pit, attr = criterion(logits, labels, existence_probs, CONFIG["num_speakers"])

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item(); pit_total += pit.item(); attr_total += attr.item()
        if (batch_idx + 1) % 10 == 0:
            print(f"  Batch {batch_idx+1}/{len(loader)} | Loss: {loss.item():.4f} "
                  f"(PIT: {pit.item():.4f} ATT: {attr.item():.4f})", end="\r")
    print()
    n = len(loader)
    return total_loss/n, pit_total/n, attr_total/n


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = pit_total = attr_total = 0
    with torch.no_grad():
        for feats, labels in loader:
            feats  = feats.to(device)
            labels = labels.to(device)
            logits, existence_probs = model(feats, num_speakers=CONFIG["num_speakers"])
            loss, pit, attr = criterion(logits, labels, existence_probs, CONFIG["num_speakers"])
            total_loss += loss.item(); pit_total += pit.item(); attr_total += attr.item()
    n = len(loader)
    return total_loss/n, pit_total/n, attr_total/n


# ============================================================
#  Vong lap huan luyen — CO RESUME (song sot qua session Colab)
# ============================================================
import os, torch

ckpt_dir  = CONFIG["checkpoint_dir"]
last_path = os.path.join(ckpt_dir, "last_model.pt")   # luu MOI epoch (de resume)
best_path = os.path.join(ckpt_dir, "best_model.pt")   # luu khi val cai thien

start_epoch    = 1
best_val_loss  = float("inf")
patience_count = 0
history = {"train_loss": [], "val_loss": [], "train_pit": [], "val_pit": [],
           "train_attr": [], "val_attr": [], "lr": []}

# --- Neu da co last_model.pt (vd: session truoc bi ngat) -> hoc tiep ---
if os.path.exists(last_path):
    ck = torch.load(last_path, map_location=device)
    model.load_state_dict(ck["model_state"])
    optimizer.load_state_dict(ck["optimizer"])
    scheduler.load_state_dict(ck["scheduler"])
    start_epoch    = ck["epoch"] + 1
    best_val_loss  = ck["best_val_loss"]
    patience_count = ck["patience_count"]
    history        = ck["history"]
    print(f"  Resume tu epoch {ck['epoch']}  ->  bat dau epoch {start_epoch} "
          f"(best_val={best_val_loss:.4f})\n")
else:
    print("  Bat dau huan luyen tu dau...\n")

for epoch in range(start_epoch, CONFIG["num_epochs"] + 1):
    print(f"Epoch {epoch:02d}/{CONFIG['num_epochs']}")
    train_loss, train_pit, train_attr = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss,   val_pit,   val_attr   = validate(model, val_loader, criterion, device)

    scheduler.step(val_loss)
    cur_lr = optimizer.param_groups[0]["lr"]

    history["train_loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["train_pit"].append(train_pit);   history["val_pit"].append(val_pit)
    history["train_attr"].append(train_attr); history["val_attr"].append(val_attr)
    history["lr"].append(cur_lr)

    print(f"   Train Loss: {train_loss:.4f} (PIT:{train_pit:.4f} ATT:{train_attr:.4f}) | "
          f"Val Loss: {val_loss:.4f} (PIT:{val_pit:.4f} ATT:{val_attr:.4f}) | lr={cur_lr:.2e}")

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss  = val_loss
        patience_count = 0
        torch.save({"epoch": epoch, "model_state": model.state_dict(),
                    "optimizer": optimizer.state_dict(), "val_loss": val_loss,
                    "config": CONFIG}, best_path)
        print(f"   ✓ Saved BEST (val_loss={val_loss:.4f})")
    else:
        patience_count += 1
        print(f"   No improvement ({patience_count}/{CONFIG['patience']})")

    # LUU last_model.pt MOI epoch -> Google Drive (qua symlink). Khong mat khi reset session.
    torch.save({"epoch": epoch, "model_state": model.state_dict(),
                "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
                "best_val_loss": best_val_loss, "patience_count": patience_count,
                "history": history, "config": CONFIG}, last_path)

    if patience_count >= CONFIG["patience"]:
        print(f"\n  Early stopping at epoch {epoch}")
        break

print("\nTraining complete!")
print(f"  Best val loss : {best_val_loss:.4f}")




> Training Curves



In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"],   label="Val")
axes[0].set_title("Total Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_pit"], label="Train")
axes[1].plot(history["val_pit"],   label="Val")
axes[1].set_title("PIT Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

axes[2].plot(history["train_attr"], label="Train")
axes[2].plot(history["val_attr"],   label="Val")
axes[2].set_title("Attractor Loss")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.suptitle("Training History", fontsize=14)
plt.tight_layout()
import os
os.makedirs("/home/ju1ian/Documents/eend/outputs", exist_ok=True)
plt.savefig("/home/ju1ian/Documents/eend/outputs/training_curves.png")
plt.show()
print("✅ Plot saved to outputs/")

# Evaluation

In [ ]:
import torch
import sys
import os

sys.path.append("/home/ju1ian/Documents/eend/models")
for mod in ["eend", "encoder", "attractor"]:
    if mod in sys.modules:
        del sys.modules[mod]

from eend import EEND

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load checkpoint
checkpoint = torch.load(
    os.path.join(CONFIG["checkpoint_dir"], "best_model.pt"),
    map_location=device
)

# Rebuild model from saved config
saved_config = checkpoint["config"]
model = EEND(
    input_size     = saved_config["input_size"],
    hidden_size    = saved_config["hidden_size"],
    num_layers     = saved_config["num_layers"],
    dropout        = saved_config["dropout"],
    attractor_size = saved_config["attractor_size"],
).to(device)

model.load_state_dict(checkpoint["model_state"])
model.eval()

print(f" Loaded best model from epoch {checkpoint['epoch']}")
print(f"  Val loss at save : {checkpoint['val_loss']:.4f}")

In [ ]:
import numpy as np

def logits_to_labels(logits, threshold=0.5):
    """
    Convert raw model output to binary speaker activity.

    logits    : (frames, num_speakers) — raw scores
    threshold : above this = speaker is active

    Returns binary array (frames, num_speakers)
    """
    probs = torch.sigmoid(logits)           # convert to 0-1 range
    return (probs > threshold).float()


def labels_to_segments(binary_labels, frame_shift_sec=0.01):
    """
    Convert per-frame binary labels to time segments.

    binary_labels   : (frames, num_speakers)
    frame_shift_sec : seconds per frame

    Returns list of (start_sec, end_sec, speaker_id) tuples
    """
    segments = []
    num_frames, num_speakers = binary_labels.shape

    for spk in range(num_speakers):
        activity = binary_labels[:, spk].numpy()
        in_segment = False
        start = 0

        for i, active in enumerate(activity):
            if active and not in_segment:
                start = i
                in_segment = True
            elif not active and in_segment:
                segments.append((
                    start * frame_shift_sec,
                    i * frame_shift_sec,
                    f"SPK_{spk:02d}"
                ))
                in_segment = False

        # Close any open segment at the end
        if in_segment:
            segments.append((
                start * frame_shift_sec,
                num_frames * frame_shift_sec,
                f"SPK_{spk:02d}"
            ))

    return sorted(segments, key=lambda x: x[0])

print("✅ Helper functions defined")

In [ ]:
import numpy as np
import os

CACHE_DIR       = "/home/ju1ian/Documents/eend/data/processed"
FRAME_SHIFT_SEC = 0.01
THRESHOLD       = 0.5

all_predictions = []   # (frames, num_speakers) per sample
all_references  = []   # (frames, num_speakers) per sample

print("Running evaluation on validation set...")

model = model.to(device)
model.eval()
with torch.no_grad():
    for idx in val_indices:
        # Doc dac trung + nhan da cache (.npy), giong het khi huan luyen
        features   = np.load(os.path.join(CACHE_DIR, f"feat_{idx}.npy"))
        ref_labels = np.load(os.path.join(CACHE_DIR, f"label_{idx}.npy"))
        num_frames = features.shape[0]

        chunk_size  = CONFIG["chunk_size"]
        pred_chunks = []
        for start in range(0, num_frames, chunk_size):
            end   = min(start + chunk_size, num_frames)
            chunk = torch.tensor(
                features[start:end], dtype=torch.float32
            ).unsqueeze(0).to(device)
            if chunk.shape[1] < chunk_size:
                pad   = torch.zeros(1, chunk_size - chunk.shape[1], features.shape[1]).to(device)
                chunk = torch.cat([chunk, pad], dim=1)
            logits, _ = model(chunk, num_speakers=CONFIG["num_speakers"])
            pred      = logits_to_labels(logits[0].cpu())
            pred_chunks.append(pred[:end - start])

        pred_labels = torch.cat(pred_chunks, dim=0)
        all_predictions.append(pred_labels.numpy())
        all_references.append(ref_labels)

print(f"✅ Evaluated {len(val_indices)} validation samples")


In [ ]:
def frame_level_accuracy(predictions, references):
    """
    Simple frame-level metric before we compute full DER.
    Checks what % of frames are correctly labeled.
    """
    from itertools import permutations

    total_frames   = 0
    correct_frames = 0

    for pred, ref in zip(predictions, references):
        num_frames   = min(pred.shape[0], ref.shape[0])
        pred = pred[:num_frames]
        ref  = ref[:num_frames]
        num_speakers = pred.shape[1]

        # Try all permutations, pick best
        best = 0
        for perm in permutations(range(num_speakers)):
            perm_pred = pred[:, perm]
            correct   = (perm_pred == ref).all(axis=1).sum()
            best      = max(best, correct)

        correct_frames += best
        total_frames   += num_frames

    return correct_frames / total_frames * 100


acc = frame_level_accuracy(all_predictions, all_references)
print(f"✅ Frame-level accuracy : {acc:.2f}%")
print(f"   (measures % of frames where ALL speakers are correctly labeled)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, os
from itertools import permutations

OUTPUT_DIR = "/home/ju1ian/Documents/eend/outputs"   # tren Colab = symlink sang Google Drive
os.makedirs(OUTPUT_DIR, exist_ok=True)

def _align(pred, ref):
    """Can hoan vi toi uu de cot du doan khop tham chieu (truoc khi ve)."""
    best = None; bs = -1
    for p in permutations(range(pred.shape[1])):
        sc = (pred[:, p] == ref).sum()
        if sc > bs:
            bs = sc; best = p
    return pred[:, list(best)]

def plot_pred_vs_ref(pred, ref, title, out_path, window=3000):
    pred = _align(pred, ref)
    active = [s for s in range(ref.shape[1]) if ref[:, s].sum() > 0] or list(range(ref.shape[1]))
    W = min(ref.shape[0], window)
    fig, axes = plt.subplots(2, 1, figsize=(10, 3.4), sharex=True)
    for ax, mat, t in [(axes[0], ref[:W][:, active],  "Nhan tham chieu (Reference)"),
                       (axes[1], pred[:W][:, active], "Du doan cua mo hinh (Prediction)")]:
        ax.imshow(mat.T, aspect="auto", cmap="Blues", interpolation="nearest",
                  extent=[0, W * 0.01, len(active) - 0.5, -0.5], vmin=0, vmax=1)
        ax.set_yticks(range(len(active)))
        ax.set_yticklabels([f"SPK {chr(65+i)}" for i in range(len(active))])
        ax.set_title(t, fontsize=10, loc="left", pad=3)
    axes[1].set_xlabel("Thoi gian (giay)")
    fig.suptitle(title, fontsize=11, y=1.02)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved ->", out_path)

# ----- Figure cho tap VALIDATION -----
plot_pred_vs_ref(
    all_predictions[0], all_references[0],
    f"Validation - mau #{val_indices[0]}",
    os.path.join(OUTPUT_DIR, "val_prediction.png"),
)


VIII - Test on Unseen Data

In [ ]:
path ="/home/ju1ian/Documents/eend/data/processed/label_108.npy"
from pathlib import Path
print(Path(path).exists())

In [ ]:
# ── STEP 9: TEST ON UNSEEN DATA ─────────────────────────────
import torch, numpy as np, os, librosa
import matplotlib.pyplot as plt

print("🧪 Running test on unseen (test) samples...\n")

FRAME_SHIFT_SEC = 0.01
THRESHOLD       = 0.5
CACHE_DIR       = "/home/ju1ian/Documents/eend/data/processed"

def logits_to_labels(logits, threshold=0.5):
    probs = torch.sigmoid(logits)
    return (probs > threshold).float()

def labels_to_segments(binary_labels, frame_shift_sec=0.01):
    segments = []
    num_frames, num_speakers = binary_labels.shape
    for spk in range(num_speakers):
        activity   = binary_labels[:, spk].numpy()
        in_segment = False
        start      = 0
        for i, active in enumerate(activity):
            if active and not in_segment:
                start = i; in_segment = True
            elif not active and in_segment:
                segments.append((start * frame_shift_sec, i * frame_shift_sec, f"SPK_{spk:02d}"))
                in_segment = False
        if in_segment:
            segments.append((start * frame_shift_sec, num_frames * frame_shift_sec, f"SPK_{spk:02d}"))
    return sorted(segments, key=lambda x: x[0])

all_test_preds = []
all_test_refs  = []

model = model.to(device)
model.eval()
with torch.no_grad():
    for idx in test_indices:
        feat  = np.load(os.path.join(CACHE_DIR, f"feat_{idx}.npy"))
        label = np.load(os.path.join(CACHE_DIR, f"label_{idx}.npy"))
        num_frames = feat.shape[0]

        chunk_size  = CONFIG["chunk_size"]
        pred_chunks = []

        for start in range(0, num_frames, chunk_size):
            end   = min(start + chunk_size, num_frames)
            chunk = torch.tensor(feat[start:end], dtype=torch.float32).unsqueeze(0).to(device)

            # Pad last chunk if needed
            if chunk.shape[1] < chunk_size:
                pad   = torch.zeros(1, chunk_size - chunk.shape[1], 40).to(device)
                chunk = torch.cat([chunk, pad], dim=1)

            logits, _ = model(chunk, num_speakers=CONFIG["num_speakers"])
            pred      = logits_to_labels(logits[0].cpu())
            pred_chunks.append(pred[:end - start])

        pred_labels = torch.cat(pred_chunks, dim=0)
        all_test_preds.append(pred_labels.numpy())
        all_test_refs.append(label)

print(f"✅ Tested on {len(test_indices)} unseen samples")

In [ ]:
# ----- Figure cho tap TEST (unseen) -> luu vao outputs/ (Drive) -----
# Dung lai ham plot_pred_vs_ref / OUTPUT_DIR da dinh nghia o cell Validation o tren.
plot_pred_vs_ref(
    all_test_preds[0], all_test_refs[0],
    f"Test (unseen) - mau #{test_indices[0]}",
    os.path.join(OUTPUT_DIR, "test_prediction.png"),
)


IX - DER METRICS

In [ ]:
# ── STEP 10: DER METRICS ────────────────────────────────────
print("📏 Computing DER...\n")

from itertools import permutations

def align_perm(pred, ref):
    """Can hoan vi toi uu per-file truoc khi tinh DER (chuan diarization):
    chon hoan vi cot du doan khop nhan tham chieu nhieu nhat."""
    ns = pred.shape[1]; best = None; best_score = -1
    for perm in permutations(range(ns)):
        sc = (pred[:, perm] == ref).sum()
        if sc > best_score:
            best_score = sc; best = perm
    return pred[:, list(best)]

def compute_der(predictions, references, frame_shift_sec=0.01, collar=0.25):
    """
    Compute Diarization Error Rate (DER).

    DER = (Miss + False Alarm + Confusion) / Total Reference Speech

    collar : ignore errors within 0.25s of segment boundaries
             (standard in diarization evaluation)
    """
    total_ref    = 0.0
    total_miss   = 0.0
    total_fa     = 0.0
    total_conf   = 0.0

    collar_frames = int(collar / frame_shift_sec)

    for pred, ref in zip(predictions, references):
        num_frames   = min(pred.shape[0], ref.shape[0])
        pred = pred[:num_frames]
        ref  = ref [:num_frames]
        pred = align_perm(pred, ref)   # <<< can hoan vi truoc khi tinh DER

        # Build collar mask — ignore frames near segment boundaries
        collar_mask = np.zeros(num_frames, dtype=bool)
        for spk in range(ref.shape[1]):
            activity = ref[:, spk]
            for i in range(1, num_frames):
                if activity[i] != activity[i-1]:  # boundary
                    lo = max(0, i - collar_frames)
                    hi = min(num_frames, i + collar_frames)
                    collar_mask[lo:hi] = True

        valid = ~collar_mask

        for f in range(num_frames):
            if not valid[f]:
                continue

            ref_spks  = set(np.where(ref [f] > 0)[0])
            pred_spks = set(np.where(pred[f] > 0)[0])

            ref_count  = len(ref_spks)
            pred_count = len(pred_spks)

            total_ref += ref_count
            hits       = len(ref_spks & pred_spks)
            miss       = ref_count  - hits
            fa         = pred_count - hits
            conf       = min(miss, fa)
            miss      -= conf
            fa        -= conf

            total_miss += miss
            total_fa   += fa
            total_conf += conf

    if total_ref == 0:
        return 0.0, 0.0, 0.0, 0.0

    miss_rate = total_miss / total_ref * 100
    fa_rate   = total_fa   / total_ref * 100
    conf_rate = total_conf / total_ref * 100
    der       = miss_rate + fa_rate + conf_rate

    return der, miss_rate, fa_rate, conf_rate


# Compute on validation set
val_der, val_miss, val_fa, val_conf = compute_der(all_predictions, all_references)
print("── Validation Set ──────────────────────")
print(f"  DER       : {val_der:.2f}%")
print(f"  Miss      : {val_miss:.2f}%")
print(f"  False Alarm: {val_fa:.2f}%")
print(f"  Confusion : {val_conf:.2f}%")

# Compute on test set
test_der, test_miss, test_fa, test_conf = compute_der(all_test_preds, all_test_refs)
print("\n── Test Set (Unseen) ───────────────────")
print(f"  DER       : {test_der:.2f}%")
print(f"  Miss      : {test_miss:.2f}%")
print(f"  False Alarm: {test_fa:.2f}%")
print(f"  Confusion : {test_conf:.2f}%")

# Save results
results = {
    "val_der": val_der, "val_miss": val_miss, "val_fa": val_fa, "val_conf": val_conf,
    "test_der": test_der, "test_miss": test_miss, "test_fa": test_fa, "test_conf": test_conf,
}
import json
with open("/home/ju1ian/Documents/eend/outputs/der_results.json", "w") as f:
    json.dump(results, f, indent=2)
print("\n✅ Results saved to outputs/der_results.json")

DER Bar Chart

In [ ]:
# ── DER VISUALIZATION ───────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
components = ["Miss", "False Alarm", "Confusion", "Total DER"]
colors     = ["steelblue", "tomato", "seagreen", "orange"]

for ax, (title, d_miss, d_fa, d_conf, d_der) in zip(axes, [
    ("Validation Set", val_miss,  val_fa,  val_conf,  val_der),
    ("Test Set",       test_miss, test_fa, test_conf, test_der),
]):
    values = [d_miss, d_fa, d_conf, d_der]
    bars   = ax.bar(components, values, color=colors)
    ax.set_title(title)
    ax.set_ylabel("Rate (%)")
    ax.set_ylim(0, max(values) * 1.3)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val:.1f}%", ha="center", fontsize=10)

plt.suptitle("Diarization Error Rate (DER) Breakdown", fontsize=13)
plt.tight_layout()
import os
os.makedirs("/home/ju1ian/Documents/eend/outputs", exist_ok=True)
plt.savefig("/home/ju1ian/Documents/eend/outputs/der_chart.png")
plt.show()
print("✅ DER chart saved!")

# X - Dem so nguoi noi (EDA speaker counting)
Dung xac suat ton tai (existence) cua tung attractor de uoc luong so speaker.

In [ ]:
import torch

@torch.no_grad()
def estimate_num_speakers(model, feats, max_speakers=4, threshold=0.5):
    """
    feats : (batch, frames, n_mels) hoac (frames, n_mels)
    Tra ve so speaker uoc luong cho moi sample + xac suat ton tai tung attractor.
    """
    model.eval()
    if feats.dim() == 2:
        feats = feats.unsqueeze(0)
    feats = feats.to(device)

    _, existence_logits = model(feats, num_speakers=max_speakers)   # (B, S_max, 1)
    probs = torch.sigmoid(existence_logits).squeeze(-1)             # (B, S_max)
    counts = (probs > threshold).sum(dim=1)                        # (B,)
    return counts.cpu(), probs.cpu()


# Vi du tren mot chunk cua tap test
from dataset import CallHomeDataset
_ds = CallHomeDataset(test_indices, chunk_size=CONFIG["chunk_size"], augment=False)
_feat, _label = _ds[0]
_counts, _probs = estimate_num_speakers(model, _feat, max_speakers=CONFIG["num_speakers"])

print("Existence probs:", [round(x, 3) for x in _probs[0].tolist()])
print("So speaker uoc luong :", int(_counts[0]))
print("So speaker that (label):", int((_label.sum(dim=0) > 0).sum()))
